In [1]:
# =========================================================
# BLOCK 1 - INSTALL LIBRARIES
# Colab cell: run first.
# =========================================================

!pip install -q openai anthropic google-generativeai pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 8.0 MB/s eta 0:00:00


In [2]:
# =========================================================
# BLOCK 2 - IMPORTS
# =========================================================

import os
import re
import json
import time
import zipfile
from datetime import datetime, timezone

import pandas as pd

from google.colab import userdata
from google.colab import files

from openai import OpenAI
import anthropic
import google.generativeai as genai

import base64 # visual
from pathlib import Path # visual


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# =========================================================
# BLOCK 3 - API KEYS
# Store keys in Colab Secrets:
# OPENAI_API_KEY, CLAUDE_API_KEY, GEMINI_API_KEY, LLAMA_API_KEY
# =========================================================

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
CLAUDE_API_KEY = userdata.get("CLAUDE_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
LLAMA_API_KEY = userdata.get("LLAMA_API_KEY")

In [4]:
# =========================================================
# BLOCK 4 - API CLIENTS
# =========================================================

gpt_client = OpenAI(api_key=OPENAI_API_KEY)

claude_client = anthropic.Anthropic(
    api_key=CLAUDE_API_KEY
)

llama_client = OpenAI(
    api_key=LLAMA_API_KEY,
    base_url="https://api.together.xyz/v1"
)

genai.configure(api_key=GEMINI_API_KEY)

In [5]:
# =========================================================
# BLOCK 5 - SETTINGS
# R17 is kept in the text output as "not assessed in text mode".
# It is excluded from main metrics by default.
# =========================================================

BATCH_SIZE = 5
DATA_PATH = "/content/data"

PROMPT_VERSION = "text_prompt_18_indicators_2026_05_10"
TEMPERATURE = 0
MAX_TOKENS = 8192
CHUNK_SIZE = 35000

MODEL_CHUNK_SIZE = {
    "GPT": 35000,
    "CLAUDE": 35000,
    "GEMINI": 35000,
    "LLAMA": 35000
}

JSON_RETRIES = 1

TEXT_MODELS = [
    "GPT",
    "CLAUDE",
    "GEMINI",
    "LLAMA"
]

MODEL_VERSION = {
    "GPT": "gpt-4o-2024-11-20",
    "CLAUDE": "claude-sonnet-4-6",
    "GEMINI": "gemini-2.5-flash",
    "LLAMA": "meta-llama/Llama-3.3-70B-Instruct-Turbo"
}

In [8]:
# =========================================================
# BLOCK 6 - UPLOAD ZIP FILE
# Upload your data.zip containing website folders.
# Each website folder should contain privacy.txt and banner.txt.
# =========================================================

uploaded = files.upload()

Saving visual_run_1.xlsx to visual_run_1.xlsx
Saving visual_run_2.xlsx to visual_run_2.xlsx
Saving visual_run_3.xlsx to visual_run_3.xlsx


In [ ]:
# =========================================================
# BLOCK 7 - UNZIP DATA
# =========================================================

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/")

print("ZIP extracted.")

ZIP extracted.


In [9]:
# =========================================================
# BLOCK 8 - SITE METADATA FROM THE THESIS TABLE
# This is used to create metadata.json files automatically.
# Folder matching is based mainly on the domain/folder name.
# =========================================================

SITE_METADATA = {
    "aboutyou.de": {
        "site_name": "About You",
        "country": "Germany",
        "domain": "aboutyou.de",
        "business_model": "Fashion marketplace",
        "traffic_tier": "High"
    },
    "allegro.pl": {
        "site_name": "Allegro",
        "country": "Poland",
        "domain": "allegro.pl",
        "business_model": "Marketplace",
        "traffic_tier": "High"
    },
    "amazon.de": {
        "site_name": "Amazon",
        "country": "Germany",
        "domain": "amazon.de",
        "business_model": "Marketplace + direct retail",
        "traffic_tier": "High"
    },
    "answear.com": {
        "site_name": "Answear",
        "country": "Poland",
        "domain": "answear.com",
        "business_model": "Direct retail",
        "traffic_tier": "High"
    },
    "bol.com": {
        "site_name": "Bol.com",
        "country": "Netherlands",
        "domain": "bol.com",
        "business_model": "Marketplace",
        "traffic_tier": "High"
    },
    "boozt.com": {
        "site_name": "Boozt",
        "country": "Denmark",
        "domain": "boozt.com",
        "business_model": "Direct retail",
        "traffic_tier": "High"
    },
    "carrefour.fr": {
        "site_name": "Carrefour",
        "country": "France",
        "domain": "carrefour.fr",
        "business_model": "Direct retail + marketplace",
        "traffic_tier": "High"
    },
    "cdiscount.com": {
        "site_name": "Cdiscount",
        "country": "France",
        "domain": "cdiscount.com",
        "business_model": "Direct retail + marketplace",
        "traffic_tier": "High"
    },
    "decathlon.es": {
        "site_name": "Decathlon",
        "country": "Spain",
        "domain": "decathlon.es",
        "business_model": "Direct retail + marketplace",
        "traffic_tier": "High"
    },
    "douglas.de": {
        "site_name": "Douglas",
        "country": "Germany",
        "domain": "douglas.de",
        "business_model": "Direct retail + marketplace",
        "traffic_tier": "High"
    },
    "elcorteingles.es": {
        "site_name": "El Corte Ingles",
        "country": "Spain",
        "domain": "elcorteingles.es",
        "business_model": "Direct retail",
        "traffic_tier": "Mid"
    },
    "emag.ro": {
        "site_name": "eMAG",
        "country": "Romania",
        "domain": "emag.ro",
        "business_model": "Marketplace",
        "traffic_tier": "Mid"
    },
    "fnac.com": {
        "site_name": "Fnac",
        "country": "France",
        "domain": "fnac.com",
        "business_model": "Direct retail + marketplace",
        "traffic_tier": "Mid"
    },
    "mediamarkt.de": {
        "site_name": "MediaMarkt",
        "country": "Germany",
        "domain": "mediamarkt.de",
        "business_model": "Direct retail",
        "traffic_tier": "Mid"
    },
    "notino.pl": {
        "site_name": "Notino",
        "country": "Poland",
        "domain": "notino.pl",
        "business_model": "Direct retail",
        "traffic_tier": "Mid"
    },
    "otto.de": {
        "site_name": "Otto",
        "country": "Germany",
        "domain": "otto.de",
        "business_model": "Direct retail + marketplace",
        "traffic_tier": "Mid"
    },
    "smythstoys.com": {
        "site_name": "Smyths Toys",
        "country": "Ireland",
        "domain": "smythstoys.com",
        "business_model": "Direct retail",
        "traffic_tier": "Mid"
    },
    "vinted.lt": {
        "site_name": "Vinted",
        "country": "Lithuania",
        "domain": "vinted.lt",
        "business_model": "Marketplace",
        "traffic_tier": "Mid"
    },
    "zalando.com": {
        "site_name": "Zalando",
        "country": "Germany",
        "domain": "zalando.com",
        "business_model": "Fashion marketplace",
        "traffic_tier": "Mid"
    },
    "zooplus.de": {
        "site_name": "Zooplus",
        "country": "Germany",
        "domain": "zooplus.de",
        "business_model": "Direct retail",
        "traffic_tier": "Mid"
    }
}

SITE_ORDER = list(SITE_METADATA.keys())

In [10]:
# =========================================================
# BLOCK 9 - CREATE metadata.json FILES
# Run after unzipping data.
# This writes metadata.json into each matched website folder.
# =========================================================

def canonical_site_key(value):
    value = str(value).lower().strip()
    value = value.replace("https://", "").replace("http://", "")
    value = value.replace("www.", "")
    value = value.strip("/")
    return value


def discover_site_folders(path=DATA_PATH):
    found = {}

    for root, dirs, filenames in os.walk(path):
        lower_files = {name.lower() for name in filenames}

        has_policy = "privacy.txt" in lower_files
        has_banner = "banner.txt" in lower_files

        if not (has_policy and has_banner):
            continue

        folder_key = canonical_site_key(os.path.basename(root))

        if folder_key not in found:
            found[folder_key] = root
        else:
            # Prefer the shortest path if duplicated nested data folders exist.
            if len(root) < len(found[folder_key]):
                found[folder_key] = root

    return found


def write_metadata_json_files(path=DATA_PATH):
    site_folders = discover_site_folders(path)
    written = []
    unmatched = []

    for site_key, folder in site_folders.items():
        meta = SITE_METADATA.get(site_key)

        if meta is None:
            unmatched.append(site_key)
            continue

        metadata_path = os.path.join(folder, "metadata.json")

        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

        written.append({
            "site_key": site_key,
            "folder": folder,
            "metadata_path": metadata_path
        })

    print(f"metadata.json written for {len(written)} sites.")

    if unmatched:
        print("Unmatched folders:", unmatched)

    return pd.DataFrame(written)


metadata_written_df = write_metadata_json_files()
display(metadata_written_df)

metadata.json written for 0 sites.


""


In [11]:
# =========================================================
# BLOCK 10 - LOAD SITE LIST
# =========================================================

SITE_FOLDER_INDEX = {}


def refresh_site_folder_index(path=DATA_PATH):
    global SITE_FOLDER_INDEX
    SITE_FOLDER_INDEX = discover_site_folders(path)
    return SITE_FOLDER_INDEX


def load_sites(path=DATA_PATH):
    if not SITE_FOLDER_INDEX:
        refresh_site_folder_index(path)

    discovered = list(SITE_FOLDER_INDEX.keys())

    ordered = [
        site for site in SITE_ORDER
        if site in discovered
    ]

    remaining = sorted([
        site for site in discovered
        if site not in ordered
    ])

    return ordered + remaining


sites = load_sites()
print("Sites found:", len(sites))
print(sites)

Sites found: 0
[]


In [12]:
# =========================================================
# BLOCK 11 - LOAD ONE SITE
# =========================================================

def load_site(site_key):
    if not SITE_FOLDER_INDEX:
        refresh_site_folder_index()

    folder = SITE_FOLDER_INDEX[site_key]

    with open(os.path.join(folder, "privacy.txt"), encoding="utf-8") as f:
        policy = f.read()

    with open(os.path.join(folder, "banner.txt"), encoding="utf-8") as f:
        banner = f.read()

    metadata_path = os.path.join(folder, "metadata.json")

    if os.path.exists(metadata_path):
        with open(metadata_path, encoding="utf-8") as f:
            metadata = json.load(f)
    else:
        metadata = SITE_METADATA.get(site_key, {
            "site_name": site_key,
            "country": "",
            "domain": site_key,
            "business_model": "",
            "traffic_tier": ""
        })

    return policy, banner, metadata

In [13]:
# =========================================================
# BLOCK 12 - CHUNKING
# Long privacy policies are split into chunks.
# The banner is attached to every chunk.
# =========================================================

def chunk_text(text, size=CHUNK_SIZE):
    return [
        text[i:i + size]
        for i in range(0, len(text), size)
    ]

In [ ]:
# =========================================================
# BLOCK 13 - NORMALIZATION AND EVIDENCE VALIDATION
# Fixed for already-generated ChatGPT evidence.
# =========================================================

import html
import re

EVIDENCE_OK_VALUES = (
    "NO_EVIDENCE_FOUND",
    "VISUAL_INDICATOR_NOT_ASSESSED_IN_TEXT_MODE",
    "NO_VALID_MODEL_OUTPUT"
)


def fix_chatgpt_evidence_quotes(ev):
    text = html.unescape(str(ev))

    text = (
        text.replace("?", "'")
            .replace("?", "'")
            .replace("?", '"')
            .replace("?", '"')
            .replace("?", '"')
            .replace("?", '"')
            .replace("?", '"')
    )

    # 'Ok' -> "Ok", 'opt-out' -> "opt-out", ('performance') -> ("performance")
    # Apostrophes inside words are protected: don't, user's, we're.
    text = re.sub(
        r"(?<![A-Za-z0-9])'([^'\r\n]+)'(?![A-Za-z0-9])",
        r'"\1"',
        text
    )

    return text


def normalize(text, compact=False):
    text = html.unescape(str(text)).lower()

    text = (
        text.replace("?", "'")
            .replace("?", "'")
            .replace("?", '"')
            .replace("?", '"')
            .replace("?", '"')
            .replace("?", '"')
            .replace("?", '"')
    )

    text = text.replace("'", '"')
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[??????]", "-", text)
    text = re.sub(r"\s+", " ", text).strip()

    if compact:
        text = re.sub(r"[^\w]+", "", text, flags=re.UNICODE)

    return text


def split_evidence_segments(text):
    raw_segments = re.split(
        r"(?:\.\.\.|(?<=[.!?])\s+|\n+|\r+)",
        str(text)
    )

    segments = []

    for part in raw_segments:
        part = part.strip(" ;:,.\t\r\n")
        if not part:
            continue

        subparts = [part]
        if len(part) > 220 and ";" in part:
            subparts = re.split(r"\s*;\s*", part)

        for subpart in subparts:
            subpart = subpart.strip(" ;:,.\t\r\n")
            words = re.findall(r"\w+", subpart, flags=re.UNICODE)

            if len(words) >= 5 and len(normalize(subpart, compact=True)) >= 28:
                segments.append(subpart)

    return segments


def ordered_segment_match(ev, full_text):
    segments = split_evidence_segments(ev)

    if len(segments) < 2:
        return False

    full_compact = normalize(full_text, compact=True)
    position = 0
    matched = 0

    for segment in segments:
        needle = normalize(segment, compact=True)
        found = full_compact.find(needle, position)

        if found >= 0:
            matched += 1
            position = found + len(needle)

    if len(segments) <= 3:
        return matched == len(segments)

    return matched >= max(2, len(segments) - 1)


def validate_evidence(ev, full_text, indicator_id=None):
    if indicator_id == "R17":
        return "OK"

    if ev in EVIDENCE_OK_VALUES:
        return "OK"

    ev = fix_chatgpt_evidence_quotes(ev)

    if normalize(ev) in normalize(full_text):
        return "OK"

    if normalize(ev, compact=True) in normalize(full_text, compact=True):
        return "OK"

    if ordered_segment_match(ev, full_text):
        return "OK"

    return "HALLUCINATION"


In [46]:
# Duplicate old evidence-validation cell removed. Use BLOCK 13 above.


In [47]:
# =========================================================
# BLOCK 14 - INDICATOR LISTS
# Main prompt returns 18 indicators.
# R17 is text-mode placeholder and is excluded from main metrics by default.
# =========================================================

ALL_INDICATORS = [
    "R01", "R02", "R03", "R04", "R05", "R06",
    "R07", "R08", "R09", "R10", "R11", "R12",
    "R13", "R14", "R15", "R16", "R17", "R18"
]

CONSENT_INDICATORS = {
    "R01", "R02", "R04", "R05", "R06"
}

TRANSPARENCY_INDICATORS = {
    "R03", "R07", "R08", "R09", "R10",
    "R11", "R12", "R13", "R14", "R15", "R18"
}

TECHNICAL_TEXT_INDICATORS = {
    "R16"
}

TEXT_MODE_PLACEHOLDER_INDICATORS = {
    "R17"
}

In [48]:
# =========================================================
# BLOCK 15 - INDICATOR METADATA
# Groups follow the prompt.
# Severity follows "The 18 Risk Indicators.docx".
# =========================================================

INDICATOR_INFO = {
    "R01": {
        "indicator_name": "Freely given consent",
        "indicator_group": "Validity of Consent",
        "legal_basis": "GDPR Art. 4(11), Art. 7(4)",
        "severity": "High",
        "severity_weight": 3
    },
    "R02": {
        "indicator_name": "Right to withdraw",
        "indicator_group": "Validity of Consent",
        "legal_basis": "GDPR Art. 7(3)",
        "severity": "High",
        "severity_weight": 3
    },
    "R03": {
        "indicator_name": "Demonstrability of consent",
        "indicator_group": "Validity of Consent",
        "legal_basis": "GDPR Art. 7(1)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R04": {
        "indicator_name": "Specific consent (granularity)",
        "indicator_group": "Validity of Consent",
        "legal_basis": "GDPR Art. 6(1)(a), Recital 32, EDPB Guidelines 05/2020",
        "severity": "High",
        "severity_weight": 3
    },
    "R05": {
        "indicator_name": "Informed consent",
        "indicator_group": "Validity of Consent",
        "legal_basis": "GDPR Art. 4(11), Art. 13",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R06": {
        "indicator_name": "Unambiguous indication",
        "indicator_group": "Validity of Consent",
        "legal_basis": "GDPR Art. 4(11), Recital 32",
        "severity": "High",
        "severity_weight": 3
    },
    "R07": {
        "indicator_name": "Identity of controller",
        "indicator_group": "Transparency",
        "legal_basis": "GDPR Art. 13(1)(a)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R08": {
        "indicator_name": "Contact details",
        "indicator_group": "Transparency",
        "legal_basis": "GDPR Art. 13(1)(a-b)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R09": {
        "indicator_name": "Purpose specification",
        "indicator_group": "Transparency",
        "legal_basis": "GDPR Art. 13(1)(c)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R10": {
        "indicator_name": "Legal basis disclosure",
        "indicator_group": "Transparency",
        "legal_basis": "GDPR Art. 13(1)(c)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R11": {
        "indicator_name": "Third-party recipients",
        "indicator_group": "Transparency",
        "legal_basis": "GDPR Art. 13(1)(e)",
        "severity": "High",
        "severity_weight": 3
    },
    "R12": {
        "indicator_name": "International transfers",
        "indicator_group": "Transparency",
        "legal_basis": "GDPR Art. 13(1)(f)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R13": {
        "indicator_name": "Data retention period",
        "indicator_group": "Accountability and Technical Signals",
        "legal_basis": "GDPR Art. 13(2)(a)",
        "severity": "Low",
        "severity_weight": 1
    },
    "R14": {
        "indicator_name": "Data subject rights",
        "indicator_group": "Accountability and Technical Signals",
        "legal_basis": "GDPR Art. 13(2)(b-d)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R15": {
        "indicator_name": "Complaint mechanism",
        "indicator_group": "Accountability and Technical Signals",
        "legal_basis": "GDPR Art. 13(2)(d)",
        "severity": "Medium",
        "severity_weight": 2
    },
    "R16": {
        "indicator_name": "Pre-consent tracking disclosure",
        "indicator_group": "Accountability and Technical Signals",
        "legal_basis": "ePrivacy Directive Art. 5(3); GDPR Art. 6(1)",
        "severity": "High",
        "severity_weight": 3
    },
    "R17": {
        "indicator_name": "Equal prominence of choices",
        "indicator_group": "Accountability and Technical Signals",
        "legal_basis": "GDPR Recital 32; EDPB Guidelines 05/2020; EDPB Guidelines 03/2022",
        "severity": "High",
        "severity_weight": 3
    },
    "R18": {
        "indicator_name": "Layered information / clarity of information",
        "indicator_group": "Accountability and Technical Signals",
        "legal_basis": "GDPR Art. 12(1)",
        "severity": "Medium",
        "severity_weight": 2
    }
}

In [49]:
# =========================================================
# BLOCK 16 - MAIN TEXT PROMPT
# This is your attached prompt, kept as 18 indicators.
# R17 must return a text-mode placeholder.
# =========================================================

MAIN_TEXT_PROMPT = """
You are a GDPR compliance analyst performing a strict audit. Evaluate the provided website texts (cookie banner and privacy policy) against 18 GDPR compliance risk indicators.
Return ONLY a valid JSON array with exactly 18 items ordered R01 to R18.
No text, explanations, or markdown outside the JSON array.
Each item MUST have exactly this structure:
{{
  "ID": "R01",
  "Score": 1 or 0,
  "Evidence": "exact quote or NO_EVIDENCE_FOUND",
  "Interpretation": "short explanation (1-2 sentences)"
}}
...
SCORING RULES
...
For CONSENT VALIDITY indicators (R01-R06):
- Score = 1 ONLY if there is clear positive evidence of non-compliance in the text
- Score = 0 if the text is silent, ambiguous, or does not confirm the risk
- Do NOT assume non-compliance from absence of information
For TRANSPARENCY indicators (R07-R15, R18):
- Score = 1 if the required information is completely absent from the text
- Score = 0 if the information is present, even if imperfect
- Absence of required disclosure = risk present
For TECHNICAL SIGNALS (R16, R17):
- Follow the specific rules stated under each indicator below
...
GLOBAL NON-INFERENCE RULE
Only evaluate risks that are directly and explicitly observable
from the provided text or screenshots
Do NOT:
- infer hidden interface behavior
- assume the existence or absence of second-layer settings
- assume technical implementation
- infer coercion, manipulation, or illegality unless explicitly stated
- treat persuasive language alone as evidence of non-compliance
If the text is ambiguous, incomplete, or open to multiple interpretations, return Score = 0.
...
EVIDENCE RULE (STRICT)
...
- Evidence MUST be copied EXACTLY word-for-word from the input text
- Do NOT rephrase, summarise, translate, or modify the quote in any way
- The quoted text must appear verbatim in the input
- If no exact match exists - return "NO_EVIDENCE_FOUND"
- If you are not 100% certain the text appears in the input - return "NO_EVIDENCE_FOUND"
- For Score = 0 with missing information: return "NO_EVIDENCE_FOUND"
- For Score = 0 with present information: quote the relevant passage
DO NOT USE:
- Paraphrased or reconstructed sentences
- General cookie descriptions not tied to a specific indicator
- Partially matching text
- Text from your training data or general knowledge
...
INTERPRETATION RULE
...
- Briefly explain WHY the score was assigned (1-2 sentences)
- This field may be paraphrased - it is your reasoning, not a quote
- Reference the specific risk condition where relevant
...
OUTPUT COMPLETENESS
...
- Return ALL 18 indicators in order R01 through R18
- Never omit an indicator
- If an indicator cannot be assessed from the available text,
  return Score = 0 and Evidence = "NO_EVIDENCE_FOUND"
...
INDICATORS
...
GROUP 1: VALIDITY OF CONSENT (R01-R06)
Scoring rule: Score = 1 only with positive evidence of non-compliance.
R01 - Freely given consent
Risk condition: Consent is bundled with service access ('accept or leave'); the text explicitly states that access to the service or core functionality is conditional on consent to non-essential processing.
R02 - Right to withdraw
Risk condition: No withdrawal mechanism is described anywhere in the texts, or the described process for withdrawing consent appears more difficult or requires more steps than the process for giving consent.
R03 - Demonstrability of consent
Risk condition: No reference to consent record-keeping, audit trails, or the ability to demonstrate that consent was obtained is made anywhere in the policy text. R03 follows transparency logic: complete absence of any reference to consent record-keeping = Score 1.
R04 - Specific consent (granularity)
Risk condition: Multiple processing purposes (e.g., analytics, marketing, personalisation) are bundled under a single opt-in with no separate choices offered per purpose category.
Do NOT assume the absence of granular choices unless the text explicitly indicates that only a single bundled choice is available. Do not infer interface structure beyond the provided text or screenshot.
R05 - Informed consent
Risk condition: No link, reference, or pointer to full privacy information is provided to the user before or at the point of consent collection.
R06 - Unambiguous indication
Risk condition: Consent is described as resulting from continued use of the site, page scrolling, inactivity, or any action other than an explicit affirmative act.
NOTE: Detection of pre-ticked boxes in preference centres requires second-layer interface interaction and is outside the scope of this text-based analysis. Do not score based on assumed box states.
...
GROUP 2: TRANSPARENCY (R07-R15, R18)
Scoring rule: Score = 1 if required information is completely absent.
R07 - Identity of controller
Risk condition: The data controller is not identified by name anywhere in the banner or policy text.
R08 - Contact details
Risk condition: No contact details (postal address, email, or equivalent) are provided for the data controller; and, where a DPO is required, no DPO contact information is provided.
R09 - Purpose specification
Risk condition: Processing purposes are described exclusively in vague or generic terms (e.g., 'to improve our services', 'for analytics purposes') with no reasonably specific explanation of the processing purpose or the role of the data in that processing activity.
R10 - Legal basis disclosure
Risk condition: The legal basis for processing personal data is not stated anywhere in the policy; or multiple legal bases are cited inconsistently across purposes without clear mapping of which basis applies to which purpose.
R11 - Third-party recipients
Risk condition: Recipients or categories of recipients to whom personal data is disclosed are not mentioned anywhere in the policy text.
R12 - International transfers
Risk condition: International transfers of personal data outside the EU/EEA are not disclosed; or where transfers are mentioned, no applicable safeguard (e.g., adequacy decision, standard contractual clauses) is mentioned.
R13 - Data retention period
Risk condition: No retention period is stated; or only vague terms are used (e.g., 'as long as necessary', 'for a reasonable period') without any specific duration or criteria for determining duration.
Score = 1 if no retention information is provided at all, or only vague terms are used with no qualifying criteria or specific examples.
Score = 0 if vague terms are accompanied by specific timeframes, legal criteria, or purpose-specific retention periods.
R14 - Data subject rights
Risk condition: Data subject rights under GDPR (access, erasure, rectification, portability, restriction, objection) are not mentioned anywhere in the text.
R15 - Complaint mechanism
Risk condition: The right to lodge a complaint with a supervisory authority (e.g., a national data protection authority) is not mentioned anywhere in the policy text.
R18 - Clarity of information
Risk condition: Information is presented in an overly complex, technical, or inaccessible format; information is scattered without clear structure; or layered presentation (brief summary + detailed policy) is absent, forcing users to navigate a single dense document to find relevant information.
Score = 1 only if the text is evidently difficult to navigate, overly dense, or structurally inaccessible to an average user.
Do not assign Score = 1 based solely on legal or technical terminology.
Apply this indicator with caution - some complexity is inherent in legal disclosure.
Score = 0 if the text, while imperfect, is reasonably navigable.
...
GROUP 3: ACCOUNTABILITY AND TECHNICAL SIGNALS (R16-R17)
R16 - Pre-consent tracking disclosure
Risk condition: The policy or banner text explicitly describes non-essential cookies or tracking technologies as activating upon site visit or page load, without any reference to prior consent being obtained.
Scoring rule: Score = 1 ONLY if the text positively describes pre-consent activation.
Score = 0 if the text is silent or if it states that tracking requires consent.
Do NOT infer from technical descriptions alone.
NOTE: This indicator is assessed at the level of textual disclosure only. Technical verification of actual cookie firing is outside the scope of this analysis.
R17 - Equal prominence of choices
Risk condition: The reject or withdraw option appears visually less prominent than the accept option (e.g., smaller, greyed out, absent from first layer, requiring more clicks to reach).
IMPORTANT: This indicator is assessed through SCREENSHOT analysis only.
When analysing text input (no image provided), return: Score = 0
Evidence = "VISUAL_INDICATOR_NOT_ASSESSED_IN_TEXT_MODE"
Interpretation = "R17 requires visual analysis of the banner screenshot and cannot be assessed from text input alone."
...
CONSERVATIVE SCORING RULE
When uncertainty exists, assign Score = 1 only if the text contains reasonably direct evidence supporting the risk condition.
Do not assign Score = 1 based purely on speculation, assumption, or inferred hidden behaviour.
This audit identifies directly observable compliance risks, not hypothetical or inferred violations.
...
Return valid JSON only. Output must be parseable by Python json.loads().
No markdown formatting, no code blocks, no trailing commas, no comments inside the JSON array.
...
TEXT TO ANALYSE
...
[BANNER]:
{banner_text}
[POLICY]:
{privacy_policy_text}
…
CRITICAL OUTPUT RULE:
Return ONLY a JSON array.
Do NOT include any text before or after.
Do NOT include ```json or markdown.
If you violate this, the output is invalid.
"""

In [50]:
# =========================================================
# BLOCK 17 - JSON EXTRACTION AND STRUCTURE VALIDATION
# A chunk is accepted only if the model returns all R01-R18.
# =========================================================

def extract_json_array(text):
    if not text:
        return None

    clean = (
        str(text)
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    try:
        parsed = json.loads(clean)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass

    start = clean.find("[")
    end = clean.rfind("]")

    if start != -1 and end != -1 and end > start:
        candidate = clean[start:end + 1]

        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

    return None


def normalize_model_item(item):
    rid = str(item.get("ID", "")).strip()

    try:
        score = int(item.get("Score", 0))
    except Exception:
        score = 0

    score = 1 if score == 1 else 0

    if rid == "R17":
        return {
            "ID": "R17",
            "Score": 0,
            "Evidence": "VISUAL_INDICATOR_NOT_ASSESSED_IN_TEXT_MODE",
            "Interpretation": "R17 requires visual analysis of the banner screenshot and cannot be assessed from text input alone."
        }

    return {
        "ID": rid,
        "Score": score,
        "Evidence": str(item.get("Evidence", "NO_EVIDENCE_FOUND")),
        "Interpretation": str(item.get("Interpretation", ""))
    }


def validate_result_structure(parsed):
    if not isinstance(parsed, list):
        return False, []

    normalized = [
        normalize_model_item(item)
        for item in parsed
        if isinstance(item, dict)
    ]

    ids = [item["ID"] for item in normalized]

    if ids != ALL_INDICATORS:
        return False, normalized

    return True, normalized

In [51]:
# =========================================================
# BLOCK 18 - MODEL CALLS
# =========================================================

def call_gpt(prompt):
    r = gpt_client.chat.completions.create(
        model=MODEL_VERSION["GPT"],
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    return r.choices[0].message.content.strip()


def call_claude(prompt):
    r = claude_client.messages.create(
        model=MODEL_VERSION["CLAUDE"],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return r.content[0].text.strip()


def call_gemini(prompt):
    model = genai.GenerativeModel(
        MODEL_VERSION["GEMINI"]
    )

    r = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(
            temperature=TEMPERATURE,
            max_output_tokens=8192,
            response_mime_type="application/json"
        )
    )

    return r.text.strip()

def call_llama(prompt):
    r = llama_client.chat.completions.create(
        model=MODEL_VERSION["LLAMA"],
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    return r.choices[0].message.content.strip()


def call_model(model, prompt):
    if model == "GPT":
        return call_gpt(prompt)
    if model == "CLAUDE":
        return call_claude(prompt)
    if model == "GEMINI":
        return call_gemini(prompt)
    if model == "LLAMA":
        return call_llama(prompt)

    return "ERROR: unknown model"

In [52]:
# =========================================================
# BLOCK 19 - API RETRY AND JSON RETRY
# API retry handles failed requests.
# JSON retry handles malformed or incomplete JSON.
# =========================================================

def call_with_retry(model, prompt, retries=3):
    last_error = ""

    for attempt in range(retries + 1):
        try:
            out = call_model(model, prompt)

            if out:
                return out

        except Exception as e:
            last_error = str(e)
            print(f"API retry {attempt + 1}: {last_error}")

        time.sleep(3)

    return f"ERROR: {last_error}"


def call_and_parse_with_json_retry(model, prompt, json_retries=JSON_RETRIES):
    attempts = []

    for attempt_no in range(json_retries + 1):

        retry_prompt = prompt

        if attempt_no > 0:
            retry_prompt += (
                "\nReturn ONLY valid JSON array. "
                "Do not include markdown, explanations, or comments."
            )

        out = call_with_retry(model, retry_prompt)

        parsed = extract_json_array(out)
        structure_valid, normalized = validate_result_structure(parsed)

        attempts.append({
            "attempt_no": attempt_no + 1,
            "json_valid": isinstance(parsed, list),
            "structure_valid": structure_valid,
            "output_full": out
        })

        if structure_valid:
            return normalized, attempts

    return None, attempts

In [53]:
# =========================================================
# BLOCK 20 - EVIDENCE QUALITY AND AGGREGATION
# Aggregation combines chunk-level results into one site result.
# =========================================================

def evidence_quality_rank(item, full_text):
    ev = item.get("Evidence", "")

    if ev in (
        "NO_EVIDENCE_FOUND",
        "VISUAL_INDICATOR_NOT_ASSESSED_IN_TEXT_MODE",
        "NO_VALID_MODEL_OUTPUT"
    ):
        return (0, 0)

    if validate_evidence(ev, full_text) == "OK":
        return (2, len(ev))

    return (1, len(ev))


def is_better_evidence_contextual(new, old, full_text):
    if old is None:
        return True

    return evidence_quality_rank(new, full_text) > evidence_quality_rank(old, full_text)


def aggregate_text_indicator(agg, item, full_text):
    rid = item.get("ID")
    score = item.get("Score")

    if rid not in ALL_INDICATORS:
        return

    if rid == "R17":
        if rid not in agg:
            agg[rid] = item
        return

    if rid not in agg:
        agg[rid] = item
        return

    current = agg[rid]

    if rid in CONSENT_INDICATORS or rid in TECHNICAL_TEXT_INDICATORS:
        if score == 1 and (
            current["Score"] == 0
            or is_better_evidence_contextual(item, current, full_text)
        ):
            agg[rid] = item

    elif rid in TRANSPARENCY_INDICATORS:
        if score == 0 and (
            current["Score"] == 1
            or is_better_evidence_contextual(item, current, full_text)
        ):
            agg[rid] = item


def finalize_text_aggregation(agg):
    for rid in ALL_INDICATORS:
        if rid not in agg:
            if rid == "R17":
                agg[rid] = {
                    "ID": "R17",
                    "Score": 0,
                    "Evidence": "VISUAL_INDICATOR_NOT_ASSESSED_IN_TEXT_MODE",
                    "Interpretation": "R17 requires visual analysis of the banner screenshot and cannot be assessed from text input alone."
                }
            else:
                agg[rid] = {
                    "ID": rid,
                    "Score": 0,
                    "Evidence": "NO_EVIDENCE_FOUND",
                    "Interpretation": "Indicator not detected in model output."
                }

    return [
        agg[rid]
        for rid in ALL_INDICATORS
    ]

In [54]:
# =========================================================
# BLOCK 21 - FILE NAMING AND TIMESTAMP HELPERS
# =========================================================

def safe_model_name(model):
    return model.lower().replace(" ", "_")


def make_run_prefix(model, start_idx, run_type="main", run_no=1):
    batch_no = (start_idx // BATCH_SIZE) + 1
    return f"{run_type}_run_{run_no:02}_{safe_model_name(model)}_batch_{batch_no:02}"


def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

In [55]:
# =========================================================
# BLOCK 22 - RUN ONE MODEL, ONE BATCH OF 5 SITES
# This is the safest execution unit.
# =========================================================

def run_text_batch_for_model(
    model,
    start_idx=0,
    selected_sites=None,
    run_type="main",
    run_no=1,
    json_retries=JSON_RETRIES
):
    if model not in TEXT_MODELS:
        raise ValueError(f"Unknown model: {model}")

    all_sites = selected_sites if selected_sites is not None else load_sites()
    batch_sites = all_sites[start_idx:start_idx + BATCH_SIZE]
    run_prefix = make_run_prefix(model, start_idx, run_type, run_no)

    results_rows = []
    raw_rows = []
    malformed_rows = []
    log_rows = []

    for site_index, site_key in enumerate(batch_sites, start=1):
        print(f"\n===== {run_prefix} | SITE {site_index}/{len(batch_sites)}: {site_key} =====")

        policy, banner, metadata = load_site(site_key)
        full_text = policy + "\n" + banner

        chunks = chunk_text(
            policy,
            size=MODEL_CHUNK_SIZE.get(model, CHUNK_SIZE)
        )
        agg = {}

        total_chunks = len(chunks)
        successful_chunks = 0
        malformed_chunks = 0

        for chunk_id, chunk in enumerate(chunks):
            print(f"{model} | {site_key} | chunk {chunk_id + 1}/{total_chunks}")

            prompt = MAIN_TEXT_PROMPT.format(
                banner_text=banner,
                privacy_policy_text=chunk
            )

            parsed, attempts = call_and_parse_with_json_retry(
                model,
                prompt,
                json_retries=json_retries
            )

            final_attempt = attempts[-1]
            structure_valid = final_attempt["structure_valid"]

            for attempt in attempts:
                raw_rows.append({
                    "run_prefix": run_prefix,
                    "run_type": run_type,
                    "run_no": run_no,
                    "site_key": site_key,
                    "site_name": metadata.get("site_name", site_key),
                    "domain": metadata.get("domain", site_key),
                    "model": model,
                    "model_version": MODEL_VERSION[model],
                    "temperature": TEMPERATURE,
                    "prompt_version": PROMPT_VERSION,
                    "created_at_utc": utc_now_iso(),
                    "chunk_id": chunk_id,
                    "total_chunks": total_chunks,
                    "chunk_length": len(chunk),
                    "attempt_no": attempt["attempt_no"],
                    "json_valid": attempt["json_valid"],
                    "structure_valid": attempt["structure_valid"],
                    "output_size": len(attempt["output_full"] or ""),
                    "output_full": attempt["output_full"]
                })

            if not structure_valid:
                malformed_chunks += 1
                malformed_rows.append({
                    "run_prefix": run_prefix,
                    "run_type": run_type,
                    "run_no": run_no,
                    "site_key": site_key,
                    "site_name": metadata.get("site_name", site_key),
                    "domain": metadata.get("domain", site_key),
                    "model": model,
                    "chunk_id": chunk_id,
                    "total_chunks": total_chunks,
                    "attempts": len(attempts),
                    "last_output_full": final_attempt["output_full"]
                })

                pd.DataFrame(raw_rows).to_excel(f"raw_outputs_{run_prefix}.xlsx", index=False)
                pd.DataFrame(malformed_rows).to_excel(f"malformed_outputs_{run_prefix}.xlsx", index=False)
                continue

            successful_chunks += 1

            for item in parsed:
                aggregate_text_indicator(agg, item, full_text)

            pd.DataFrame(raw_rows).to_excel(f"raw_outputs_{run_prefix}.xlsx", index=False)
            if model == "LLAMA":
                time.sleep(10)
            else:
                time.sleep(1)


        final_items = finalize_text_aggregation(agg)
        chunk_coverage = successful_chunks / total_chunks if total_chunks else 0

        for item in final_items:
            rid = item["ID"]
            meta = INDICATOR_INFO[rid]
            evidence_check = validate_evidence(item["Evidence"], full_text)

            results_rows.append({
                "run_prefix": run_prefix,
                "run_type": run_type,
                "run_no": run_no,
                "site_key": site_key,
                "site_name": metadata.get("site_name", site_key),
                "domain": metadata.get("domain", site_key),
                "country": metadata.get("country", ""),
                "business_model": metadata.get("business_model", ""),
                "traffic_tier": metadata.get("traffic_tier", ""),
                "model": model,
                "model_version": MODEL_VERSION[model],
                "temperature": TEMPERATURE,
                "prompt_version": PROMPT_VERSION,
                "created_at_utc": utc_now_iso(),
                "indicator_id": rid,
                "indicator_name": meta["indicator_name"],
                "indicator_group": meta["indicator_group"],
                "legal_basis": meta["legal_basis"],
                "severity": meta["severity"],
                "severity_weight": meta["severity_weight"],
                "llm_score": item["Score"],
                "weighted_score": item["Score"] * meta["severity_weight"],
                "evidence_quote": item["Evidence"],
                "llm_interpretation": item["Interpretation"],
                "evidence_check": evidence_check,
                "total_chunks": total_chunks,
                "successful_chunks": successful_chunks,
                "malformed_chunks": malformed_chunks,
                "chunk_coverage": chunk_coverage,
                "human_score": "",
                "validation_category": "",
                "validation_note": ""
            })

        log_rows.append({
            "run_prefix": run_prefix,
            "site_key": site_key,
            "site_name": metadata.get("site_name", site_key),
            "domain": metadata.get("domain", site_key),
            "model": model,
            "status": "SUCCESS",
            "total_chunks": total_chunks,
            "successful_chunks": successful_chunks,
            "malformed_chunks": malformed_chunks,
            "chunk_coverage": chunk_coverage
        })

        pd.DataFrame(results_rows).to_excel(f"results_{run_prefix}.xlsx", index=False)
        pd.DataFrame(raw_rows).to_excel(f"raw_outputs_{run_prefix}.xlsx", index=False)
        pd.DataFrame(malformed_rows).to_excel(f"malformed_outputs_{run_prefix}.xlsx", index=False)
        pd.DataFrame(log_rows).to_excel(f"run_log_{run_prefix}.xlsx", index=False)

        print(f"Checkpoint saved for {site_key}. Chunk coverage: {chunk_coverage:.2%}")

    return (
        pd.DataFrame(results_rows),
        pd.DataFrame(raw_rows),
        pd.DataFrame(malformed_rows),
        pd.DataFrame(log_rows)
    )

In [ ]:
# =========================================================
# BLOCK 23 - MAIN ANALYSIS: 20 SITES
# Run manually by model and batch.
# Do not run all at once unless you are sure about cost/time.
# =========================================================

def run_main_20_sites_for_model(model):
    outputs = []

    for start_idx in [0, 5, 10, 15]:
        outputs.append(
            run_text_batch_for_model(
                model=model,
                start_idx=start_idx,
                run_type="main",
                run_no=1
            )
        )

# Manual examples:
# run_text_batch_for_model("CLAUDE", start_idx=10, run_type="main", run_no=1)
# run_text_batch_for_model("GEMINI", start_idx=5, run_type="main", run_no=1)
# run_text_batch_for_model("GPT", start_idx=10, run_type="main", run_no=1)
# run_text_batch_for_model("LLAMA", start_idx=15, run_type="main", run_no=1)

# Repeat the same four calls for:
# "CLAUDE", "GEMINI", "LLAMA"

In [56]:
# =========================================================
# BLOCK 24 - REPEAT RUNS: 10 SITES x 2 RUNS
# This is independent from the main run.
# =========================================================

def get_repeat_sites(n=10):

    selected_domains = [
        "amazon.de",
        "zalando.com",
        "allegro.pl",
        "mediamarkt.de",
        "fnac.com",
        "notino.pl",
        "emag.ro",
        "aboutyou.de",
        "zooplus.de",
        "decathlon.es"
    ]

    all_sites = load_sites()

    return [
        site for site in all_sites
        if site in selected_domains
    ]


def run_repeat_10_sites_for_model(model, repeat_sites=None):
    sites = repeat_sites if repeat_sites is not None else get_repeat_sites(10)
    outputs = []

    for run_no in [1, 2]:
        for start_idx in [0, 5]:
            outputs.append(
                run_text_batch_for_model(
                    model=model,
                    start_idx=start_idx,
                    selected_sites=sites,
                    run_type="repeat",
                    run_no=run_no
                )
            )

    return outputs

# Manual examples:
# repeat_sites = get_repeat_sites(10)

# run_text_batch_for_model("GEMINI", start_idx=0, selected_sites=repeat_sites, run_type="repeat", run_no=1)
# run_text_batch_for_model("GEMINI", start_idx=5, selected_sites=repeat_sites, run_type="repeat", run_no=1)
# run_text_batch_for_model("GEMINI", start_idx=0, selected_sites=repeat_sites, run_type="repeat", run_no=2)
# run_text_batch_for_model("GEMINI", start_idx=5, selected_sites=repeat_sites, run_type="repeat", run_no=2)

# Repeat for "CLAUDE", "GEMINI", "LLAMA".

In [57]:
# =========================================================
# BLOCK 25 - VISUAL PROMPT
# =========================================================

VISUAL_PROMPT = """
You are a GDPR compliance analyst evaluating the visual design of a cookie consent banner.

Evaluate ONLY indicator R17: Equal prominence of choices.

R17 risk condition:
The option to reject, decline, refuse, manage, or withdraw consent is visually less prominent or less immediately available than the option to accept consent.

Use ONLY the provided screenshot. Do not use the privacy policy text. Do not infer hidden behaviour unless it is directly visible in the screenshot.

Assess the first visible layer of the consent interface. Focus on:
- whether an accept option is visible;
- whether a reject or decline option is visible at the same layer;
- whether accept and reject/decline options have comparable size, colour contrast, placement, and visual emphasis;
- whether the user must click "manage", "settings", "more options", or a similar control before being able to reject;
- whether the reject/decline option is greyed out, visually hidden, smaller, lower contrast, or placed less prominently than accept.

Scoring:
- Score = 1 if accepting is visually easier, more prominent, or more immediate than rejecting/declining.
- Score = 1 if accept is available on the first layer but rejection requires opening settings or another layer.
- Score = 1 if accept and reject/decline options are both present on the first layer but differ significantly in colour, fill style (solid vs
  outline), or contrast in a way that draws attention to accept over reject.
- Score = 1 if the reject/decline option is present but clearly less prominent than accept.
- Score = 0 if accept and reject/decline are both available on the same layer and have broadly comparable visual prominence.
- Score = 0 if there is no clear accept option.
- Score = 0 if the screenshot does not contain enough visible information to assess the banner.


Evidence rules:
Because this is visual analysis, Evidence should not be a privacy-policy quote.
Evidence must briefly describe the directly visible visual facts.

If the screenshot is ambiguous or partially cropped, do not infer missing elements.

Return ONLY valid JSON:
{
  "ID": "R17",
  "Score": 1 or 0,
  "Evidence": "brief directly visible visual evidence",
  "Interpretation": "short explanation"
}
"""

In [58]:
# =========================================================
# BLOCK 26 - ENCODE IMAGE
# =========================================================

def encode_image(image_path):

    with open(image_path, "rb") as image_file:

        return base64.b64encode(
            image_file.read()
        ).decode("utf-8")


In [59]:
# =========================================================
# BLOCK 27 - JSON PARSER
# =========================================================

def parse_json(text):

    text = text.strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")

    return json.loads(text)


In [60]:
# =========================================================
# BLOCK 28 - GPT, CLAUDE, GEMINI VISION
# =========================================================

def analyze_gpt(image_path):

    base64_image = encode_image(image_path)

    response = openai_client.chat.completions.create(
        model=GPT_MODEL,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": VISUAL_PROMPT
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    }
                ]
            }
        ]
    )

    return response.choices[0].message.content

def analyze_claude(image_path):

    base64_image = encode_image(image_path)

    response = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1000,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": base64_image
                        }
                    },
                    {
                        "type": "text",
                        "text": VISUAL_PROMPT
                    }
                ]
            }
        ]
    )

    return response.content[0].text

def analyze_gemini(image_path):

    model = genai.GenerativeModel(GEMINI_MODEL)

    uploaded_image = {
        "mime_type": "image/png",
        "data": Path(image_path).read_bytes()
    }

    response = model.generate_content(
        [
            VISUAL_PROMPT,
            uploaded_image
        ],
        generation_config={
            "temperature": 0
        }
    )

    return response.text


In [61]:
# =========================================================
# BLOCK 29 - MODEL ROUTER
# =========================================================

def run_visual_model(model_name, image_path):

    if model_name == "GPT":
        return analyze_gpt(image_path)

    elif model_name == "CLAUDE":
        return analyze_claude(image_path)

    elif model_name == "GEMINI":
        return analyze_gemini(image_path)

    else:
        raise ValueError(f"Unknown model: {model_name}")


In [ ]:
# DO NOT RUN
# =========================================================
# BLOCK 30 - MAIN LOOP
# =========================================================

MODELS = [
    "GEMINI"
]

N_RUNS = 3

DATA_FOLDER = Path("data")

site_folders = sorted([
    f for f in DATA_FOLDER.iterdir()
    if f.is_dir()
])

for run_no in range(1, N_RUNS + 1):

    print(f"\n################################")
    print(f"STARTING RUN {run_no}")
    print(f"################################")

    run_results = []

    for model_name in MODELS:

        print(f"\n==============================")
        print(f"RUNNING MODEL: {model_name}")
        print(f"==============================")

        for site_folder in site_folders:

            domain = site_folder.name

            image_path = site_folder / "banner.png"

            print(f"\nRUN {run_no} | {model_name} | {domain}")

            if not image_path.exists():

                print("banner.png not found")

                run_results.append({
                    "model": model_name,
                    "domain": domain,
                    "status": "NO_IMAGE"
                })

                continue

            try:

                raw_output = run_visual_model(
                    model_name,
                    image_path
                )

                parsed = parse_json(raw_output)

                parsed["model"] = model_name
                parsed["domain"] = domain
                parsed["run_no"] = run_no
                parsed["status"] = "SUCCESS"

                run_results.append(parsed)

                print("SUCCESS")

            except Exception as e:

                print(f"ERROR: {e}")

                run_results.append({
                    "model": model_name,
                    "domain": domain,
                    "run_no": run_no,
                    "status": "ERROR",
                    "error": str(e)
                })

            time.sleep(2)


In [62]:
# =========================================================
# BLOCK 31 - SAVE VISUAL RESULTS
# =========================================================

run_df = pd.DataFrame(run_results)

filename = f"visual_run_{run_no}.xlsx"

run_df.to_excel(filename, index=False)

print(f"\nSaved: {filename}")

NameError: name 'run_results' is not defined

In [29]:
# =========================================================
# BLOCK 32 - ZIP ALL EXCEL FILES IN NOTEBOOK
# =========================================================

import zipfile
from pathlib import Path
from google.colab import files

zip_filename = "all_excel_files.zip"

excel_files = list(Path(".").glob("*.xlsx"))

with zipfile.ZipFile(zip_filename, "w") as zipf:

    for file in excel_files:

        zipf.write(file)

print("Added files:")

for file in excel_files:
    print(file.name)

files.download(zip_filename)


Added files:
run_log_repeat_run_01_llama_batch_02.xlsx
raw_outputs_main_run_01_llama_batch_01.xlsx
run_log_main_run_01_llama_batch_02.xlsx
malformed_outputs_main_run_01_gpt_batch_04.xlsx
run_log_repeat_run_02_claude_batch_01.xlsx
run_log_repeat_run_01_llama_batch_01.xlsx
results_main_run_01_llama_batch_02.xlsx
results_repeat_run_02_claude_batch_02.xlsx
run_log_main_run_01_llama_batch_01.xlsx
malformed_outputs_repeat_run_02_claude_batch_02.xlsx
raw_outputs_repeat_run_01_claude_batch_01.xlsx
malformed_outputs_repeat_run_02_llama_batch_01.xlsx
run_log_repeat_run_01_gemini_batch_04.xlsx
raw_outputs_main_run_01_gpt_batch_02.xlsx
run_log_repeat_run_01_gpt_batch_01.xlsx
raw_outputs_main_run_01_gemini_batch_03.xlsx
run_log_repeat_run_01_gemini_batch_01.xlsx
raw_outputs_repeat_run_02_gemini_batch_01.xlsx
raw_outputs_repeat_run_02_gemini_batch_02.xlsx
results_main_run_01_gpt_batch_04.xlsx
run_log_main_run_01_claude_batch_01.xlsx
malformed_outputs_repeat_run_01_gemini_batch_01.xlsx
run_log_main_r

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================================================
# BLOCK 32A - UPLOAD READY EXCEL RESULT FILES
# Use this instead of running model/API blocks again.
# Upload:
# - results_main_run_01_*_batch_*.xlsx OR results_main_all_text.xlsx
# - repeat_run files if needed
# - visual_run_1.xlsx, visual_run_2.xlsx, visual_run_3.xlsx if you use R17 replacement
# =========================================================

from google.colab import files

uploaded_ready_results = files.upload()

print("Uploaded ready result files:")
for filename in uploaded_ready_results.keys():
    print("-", filename)


In [66]:
# =========================================================
# BLOCK 33 - COMBINE MAIN RESULTS EXCEL FILES
# Use after uploading ready result Excel files.
# Works with separate batch files OR an already combined file.
# =========================================================

from pathlib import Path

if "TEXT_MODELS" not in globals():
    TEXT_MODELS = ["GPT", "CLAUDE", "GEMINI", "LLAMA"]

if "safe_model_name" not in globals():
    def safe_model_name(model):
        return str(model).lower().replace(" ", "_")


def combine_excel_files(file_paths):
    frames = [
        pd.read_excel(path)
        for path in file_paths
    ]

    return pd.concat(frames, ignore_index=True)


def expected_main_result_files():
    files_list = []

    for model in TEXT_MODELS:
        for batch_no in [1, 2, 3, 4]:
            files_list.append(
                f"results_main_run_01_{safe_model_name(model)}_batch_{batch_no:02}.xlsx"
            )

    return files_list


def expected_main_raw_files():
    files_list = []

    for model in TEXT_MODELS:
        for batch_no in [1, 2, 3, 4]:
            files_list.append(
                f"raw_outputs_main_run_01_{safe_model_name(model)}_batch_{batch_no:02}.xlsx"
            )

    return files_list


def choose_existing_files(combined_file, expected_files, label):
    if all(Path(path).exists() for path in expected_files):
        return expected_files

    if Path(combined_file).exists():
        return [combined_file]

    missing = [path for path in expected_files if not Path(path).exists()]
    raise FileNotFoundError(
        f"Missing {label} files. Upload either {combined_file} or all expected batch files. "
        f"Missing batch files: {missing}"
    )


def combine_main_results():
    return combine_excel_files(
        choose_existing_files(
            "results_main_all_text.xlsx",
            expected_main_result_files(),
            "main result"
        )
    )


def combine_main_raw_outputs():
    return combine_excel_files(
        choose_existing_files(
            "raw_outputs_main_all.xlsx",
            expected_main_raw_files(),
            "main raw output"
        )
    )


In [67]:
# =========================================================
# BLOCK 34 - COMBINE REPEAT RUN FILES
# Functions only. Actual combine + revalidation happens in BLOCK 33A.
# =========================================================


def expected_repeat_result_files(run_no):
    files_list = []

    for model in TEXT_MODELS:
        for batch_no in [1, 2]:
            files_list.append(
                f"results_repeat_run_{run_no}_{safe_model_name(model)}_batch_{batch_no:02}.xlsx"
            )

    return files_list


def combine_repeat_results(run_no):
    return combine_excel_files(
        choose_existing_files(
            f"repeat_run_{run_no}_combined.xlsx",
            expected_repeat_result_files(run_no),
            f"repeat run {run_no}"
        )
    )


In [ ]:
# =========================================================
# BLOCK 33A - REPAIR HALLUCINATION CHECKS AFTER UPLOAD
# Use this when model/API outputs already exist as Excel files.
# It keeps existing OK values and rechecks only existing HALLUCINATION rows.
# =========================================================

import re
from pathlib import Path

if "DATA_PATH" not in globals():
    DATA_PATH = "/content/data"


def _safe_read_text(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return Path(path).read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    return Path(path).read_text(errors="ignore")


def _key(value):
    value = str(value or "").lower().strip()
    value = value.replace("https://", "").replace("http://", "")
    value = value.replace("www.", "")
    value = value.split("/")[0]
    return re.sub(r"[^a-z0-9]+", "", value)


def find_site_folder_for_row(row, data_path=DATA_PATH):
    roots = [Path(data_path), Path("/content")]
    candidates = [
        _key(row.get("site_key", "")),
        _key(row.get("domain", "")),
        _key(row.get("site_name", ""))
    ]
    candidates = [c for c in candidates if c]

    folders = []
    for root in roots:
        if root.exists():
            folders.extend([p for p in root.rglob("*") if p.is_dir()])

    for folder in folders:
        folder_key = _key(folder.name)
        if folder_key in candidates:
            return folder

    for folder in folders:
        folder_key = _key(folder.name)
        if any(c in folder_key or folder_key in c for c in candidates):
            return folder

    return None


def full_text_for_row(row, data_path=DATA_PATH):
    folder = find_site_folder_for_row(row, data_path=data_path)
    if folder is None:
        return None

    policy_files = list(folder.glob("privacy.txt")) + list(folder.glob("*privacy*.txt"))
    banner_files = list(folder.glob("banner.txt")) + list(folder.glob("*banner*.txt"))

    policy = _safe_read_text(policy_files[0]) if policy_files else ""
    banner = _safe_read_text(banner_files[0]) if banner_files else ""

    full_text = policy + "\n" + banner
    return full_text if full_text.strip() else None


def repair_evidence_checks_df(results_df, data_path=DATA_PATH):
    df = results_df.copy()

    if "evidence_quote" not in df.columns:
        raise ValueError("Column evidence_quote not found in results file.")

    if "evidence_check" not in df.columns:
        raise ValueError("Column evidence_check not found. Use the original result files, not the broken revalidated file.")

    full_text_cache = {}
    hallucination_to_ok = 0
    missing_source_kept_old = 0

    for idx, row in df.iterrows():
        evidence = row.get("evidence_quote", "")
        fixed_evidence = fix_chatgpt_evidence_quotes(evidence)
        df.at[idx, "evidence_quote"] = fixed_evidence

        old_status = str(row.get("evidence_check", ""))
        indicator_id = row.get("indicator_id", "")

        if indicator_id == "R17":
            df.at[idx, "evidence_check"] = "OK"
            continue

        # Preserve old OK values. Only repair rows already marked as hallucination.
        if old_status.upper() != "HALLUCINATION":
            df.at[idx, "evidence_check"] = "OK" if old_status.upper() == "OK" else old_status
            continue

        cache_key = (
            str(row.get("site_key", "")),
            str(row.get("domain", "")),
            str(row.get("site_name", ""))
        )

        if cache_key not in full_text_cache:
            full_text_cache[cache_key] = full_text_for_row(row, data_path=data_path)

        full_text = full_text_cache[cache_key]

        if full_text is None:
            missing_source_kept_old += 1
            df.at[idx, "evidence_check"] = old_status
            continue

        new_status = validate_evidence(
            fixed_evidence,
            full_text,
            indicator_id=indicator_id
        )

        if new_status == "OK":
            hallucination_to_ok += 1

        df.at[idx, "evidence_check"] = new_status

    print("Rows:", len(df))
    print("HALLUCINATION -> OK:", hallucination_to_ok)
    print("Missing source rows kept with old status:", missing_source_kept_old)
    print(df["evidence_check"].value_counts(dropna=False))

    return df


def combine_and_repair(file_paths, output_file, data_path=DATA_PATH):
    combined = combine_excel_files(file_paths)
    repaired = repair_evidence_checks_df(combined, data_path=data_path)
    repaired.to_excel(output_file, index=False)
    print("Saved:", output_file)
    return repaired


def combine_and_repair_if_available(combined_file, expected_files, label, output_file):
    try:
        files_to_use = choose_existing_files(combined_file, expected_files, label)
    except FileNotFoundError as err:
        print("Skipped", label + ":", err)
        return None

    return combine_and_repair(files_to_use, output_file)


main_all_results = combine_and_repair_if_available(
    "results_main_all_text.xlsx",
    expected_main_result_files(),
    "main result",
    "results_main_all_text_revalidated.xlsx"
)

repeat_run_01 = combine_and_repair_if_available(
    "repeat_run_01_combined.xlsx",
    expected_repeat_result_files("01"),
    "repeat run 01",
    "repeat_run_01_combined_revalidated.xlsx"
)

repeat_run_02 = combine_and_repair_if_available(
    "repeat_run_02_combined.xlsx",
    expected_repeat_result_files("02"),
    "repeat run 02",
    "repeat_run_02_combined_revalidated.xlsx"
)


In [68]:
# =========================================================
# BLOCK 35 - FUNCTION TO REPLACE ONLY R17 VALUES
# WITHOUT CHANGING MAIN COLUMN NAMES
# =========================================================

def replace_r17_values(
    target_df,
    visual_df
):

    # keep only R17 rows from visual analysis
    visual_r17 = visual_df[
        visual_df["ID"] == "R17"
    ].copy()

    # rename visual columns temporarily
    visual_r17 = visual_r17.rename(columns={

        "Score":
            "new_llm_score",

        "Evidence":
            "new_evidence_quote",

        "Interpretation":
            "new_llm_interpretation"
    })

    # merge ONLY needed columns
    merged = target_df.merge(

        visual_r17[[
            "domain",
            "model",
            "new_llm_score",
            "new_evidence_quote",
            "new_llm_interpretation"
        ]],

        on=[
            "domain",
            "model"
        ],

        how="left"
    )

    # mask ONLY R17 rows in main dataset
    r17_mask = (
        merged["indicator_id"] == "R17"
    )

    # replace values ONLY for R17
    merged.loc[
        r17_mask,
        "llm_score"
    ] = merged.loc[
        r17_mask,
        "new_llm_score"
    ]

    merged.loc[
        r17_mask,
        "evidence_quote"
    ] = merged.loc[
        r17_mask,
        "new_evidence_quote"
    ]

    merged.loc[
        r17_mask,
        "llm_interpretation"
    ] = merged.loc[
        r17_mask,
        "new_llm_interpretation"
    ]

    # remove temporary helper columns
    merged = merged.drop(columns=[

        "new_llm_score",

        "new_evidence_quote",

        "new_llm_interpretation"
    ])

    return merged

In [69]:
# =========================================================
# BLOCK 36 - CREATE NEW MAINRESULTS AFTER REVALIDATION
# This uses the corrected hallucination checks from BLOCK 33A.
# R17 is replaced only if visual_run_1.xlsx is uploaded.
# =========================================================

from pathlib import Path
from google.colab import files

main_results = pd.read_excel(
    "results_main_all_text_revalidated.xlsx"
)

if Path("visual_run_1.xlsx").exists():
    visual_run_01 = pd.read_excel(
        "visual_run_1.xlsx"
    )

    main_results_updated = replace_r17_values(
        target_df=main_results,
        visual_df=visual_run_01
    )
else:
    print("visual_run_1.xlsx not found. MainResults will keep text-mode R17 values.")
    main_results_updated = main_results.copy()

main_results_updated.to_excel(
    "MainResults_revalidated.xlsx",
    index=False
)

# Keep this old filename too, so the later metrics blocks can run unchanged.
main_results_updated.to_excel(
    "results_main_all_with_visual_r17.xlsx",
    index=False
)

files.download(
    "MainResults_revalidated.xlsx"
)


# Optional repeat outputs for repeat-stability analysis.
if Path("repeat_run_01_combined_revalidated.xlsx").exists():
    repeat_run_01 = pd.read_excel(
        "repeat_run_01_combined_revalidated.xlsx"
    )

    if Path("visual_run_2.xlsx").exists():
        visual_run_02 = pd.read_excel(
            "visual_run_2.xlsx"
        )
        updated_repeat_run_01 = replace_r17_values(
            target_df=repeat_run_01,
            visual_df=visual_run_02
        )
    else:
        updated_repeat_run_01 = repeat_run_01.copy()

    updated_repeat_run_01.to_excel(
        "repeat_run_01_with_visual_r17.xlsx",
        index=False
    )

    files.download(
        "repeat_run_01_with_visual_r17.xlsx"
    )

if Path("repeat_run_02_combined_revalidated.xlsx").exists():
    repeat_run_02 = pd.read_excel(
        "repeat_run_02_combined_revalidated.xlsx"
    )

    if Path("visual_run_3.xlsx").exists():
        visual_run_03 = pd.read_excel(
            "visual_run_3.xlsx"
        )
        updated_repeat_run_02 = replace_r17_values(
            target_df=repeat_run_02,
            visual_df=visual_run_03
        )
    else:
        updated_repeat_run_02 = repeat_run_02.copy()

    updated_repeat_run_02.to_excel(
        "repeat_run_02_with_visual_r17.xlsx",
        index=False
    )

    files.download(
        "repeat_run_02_with_visual_r17.xlsx"
    )

print("DONE")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DONE


In [70]:
# =========================================================
# BLOCK 37 - RISK TIER AND FLEISS KAPPA HELPERS
# =========================================================

def add_risk_tier(score):

    if pd.isna(score):
        return "No data"

    # =====================================================
    # Risk tier thresholds based on thesis framework
    # =====================================================

    if score >= 30:
        return "Critical"

    if score >= 20:
        return "High Risk"

    if score >= 10:
        return "Medium Risk"

    return "Low Risk"


def fleiss_kappa_binary(ratings_df):

    # remove rows with missing values
    clean = ratings_df.dropna(axis=0)

    if clean.empty:
        return pd.NA

    # number of raters/models
    n_raters = clean.shape[1]

    if n_raters < 2:
        return pd.NA

    # convert safely to numeric
    values = clean.apply(
        pd.to_numeric,
        errors="coerce"
    ).dropna(axis=0)

    if values.empty:
        return pd.NA

    # =====================================================
    # Binary agreement counts
    # Assumes ONLY 0/1 labels
    # =====================================================

    n_0 = (values == 0).sum(axis=1)
    n_1 = (values == 1).sum(axis=1)

    # observed agreement per item
    p_i = (
        (
            n_0 * (n_0 - 1)
            +
            n_1 * (n_1 - 1)
        )
        /
        (
            n_raters * (n_raters - 1)
        )
    )

    # mean observed agreement
    p_bar = p_i.mean()

    # category proportions
    p0 = n_0.sum() / (len(values) * n_raters)
    p1 = n_1.sum() / (len(values) * n_raters)

    # expected agreement by chance
    p_e = p0**2 + p1**2

    # avoid division by zero
    if p_e == 1:
        return pd.NA

    # Fleiss' Kappa
    return (
        (p_bar - p_e)
        /
        (1 - p_e)
    )

In [71]:
# Duplicate old BLOCK 38 removed. Use the human-validation aware BLOCK 38 below.


In [72]:
# =========================================================
# BLOCK 38 - PYTHON METRICS
# Human-validation aware main metrics.
# R17 INCLUDED; LLAMA excluded ONLY for R17.
# =========================================================

if "EVIDENCE_OK_VALUES" not in globals():
    EVIDENCE_OK_VALUES = (
        "NO_EVIDENCE_FOUND",
        "VISUAL_INDICATOR_NOT_ASSESSED_IN_TEXT_MODE",
        "NO_VALID_MODEL_OUTPUT"
    )


def normalize_validation_category(value, evidence_check=None, evidence_quote=None, llm_score=None):
    text = str(value).strip().upper()
    text = text.replace("-", " ").replace("_", " ")
    text = " ".join(text.split())

    if text in ("VC", "VERIFIED", "VERIFIED CLASSIFICATION", "VALID", "CORRECT"):
        return "VC"

    if text in ("MC", "MISCLASSIFICATION", "MISCLASSIFIED", "WRONG CLASSIFICATION"):
        return "MC"

    if text in ("H", "HALLUCINATION", "HALLUCINATED"):
        return "H"

    if text in ("AE", "ABSENT EVIDENCE", "NO EVIDENCE", "NO EVIDENCE FOUND", "JUSTIFIED ABSENT EVIDENCE"):
        return "AE"

    # Fallback only for rows that have not yet been manually categorized.
    evidence_status = str(evidence_check).strip().upper()
    evidence_text = str(evidence_quote).strip().upper()

    if evidence_status == "HALLUCINATION":
        return "H"

    if evidence_text in EVIDENCE_OK_VALUES:
        return "AE"

    if evidence_status == "OK":
        return "VC"

    return pd.NA


def add_validation_columns(df):
    df = df.copy()

    if "validation_category" not in df.columns:
        df["validation_category"] = pd.NA

    if "human_score" not in df.columns:
        df["human_score"] = pd.NA

    df["validation_category_norm"] = df.apply(
        lambda row: normalize_validation_category(
            row.get("validation_category", pd.NA),
            evidence_check=row.get("evidence_check", pd.NA),
            evidence_quote=row.get("evidence_quote", pd.NA),
            llm_score=row.get("llm_score", pd.NA)
        ),
        axis=1
    )

    df["human_score_num"] = pd.to_numeric(
        df["human_score"],
        errors="coerce"
    )

    df["has_human_score"] = df["human_score_num"].isin([0, 1])

    df["human_agreement"] = pd.NA
    mask = df["has_human_score"]
    df.loc[mask, "human_agreement"] = (
        df.loc[mask, "llm_score"]
        ==
        df.loc[mask, "human_score_num"]
    )

    return df


def summarize_hallucination_rate(df, group_cols):
    grouped = (
        df.groupby(group_cols, dropna=False)
        ["validation_category_norm"]
        .value_counts(dropna=False)
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["VC", "MC", "H", "AE"]:
        if col not in grouped.columns:
            grouped[col] = 0

    grouped["validation_denominator"] = (
        grouped["VC"]
        + grouped["MC"]
        + grouped["H"]
    )

    grouped["hallucination_rate"] = (
        grouped["H"]
        / grouped["validation_denominator"].replace(0, pd.NA)
    )

    grouped["groundedness_rate"] = (
        1 - grouped["hallucination_rate"]
    )

    return grouped


def summarize_human_alignment(df, group_cols):
    human = df[df["has_human_score"]].copy()

    if human.empty:
        return pd.DataFrame(
            columns=group_cols + [
                "human_validated_rows",
                "human_agreements",
                "human_disagreements",
                "human_alignment_rate"
            ]
        )

    summary = (
        human.groupby(group_cols, dropna=False)
        .agg(
            human_validated_rows=("human_agreement", "count"),
            human_agreements=("human_agreement", "sum")
        )
        .reset_index()
    )

    summary["human_disagreements"] = (
        summary["human_validated_rows"]
        - summary["human_agreements"]
    )

    summary["human_alignment_rate"] = (
        summary["human_agreements"]
        / summary["human_validated_rows"].replace(0, pd.NA)
    )

    return summary


def compute_python_metrics(
    results_df,
    raw_df=None
):

    df = results_df.copy()

    # Exclude LLAMA only for indicator R17.
    df = df[
        ~(
            (df["indicator_id"] == "R17")
            &
            (
                df["model"]
                .astype(str)
                .str.upper()
                .str.contains("LLAMA", na=False)
            )
        )
    ].copy()

    df["llm_score"] = pd.to_numeric(
        df["llm_score"],
        errors="coerce"
    ).fillna(0)

    df["severity_weight"] = pd.to_numeric(
        df["severity_weight"],
        errors="coerce"
    ).fillna(0)

    df["weighted_score"] = (
        df["llm_score"]
        *
        df["severity_weight"]
    )

    df = add_validation_columns(df)

    # =====================================================
    # SITE-LEVEL RISK METRICS
    # =====================================================

    site_model = (
        df.groupby(
            [
                "run_type",
                "run_no",
                "site_name",
                "domain",
                "model"
            ],
            as_index=False
        )
        .agg(
            weighted_risk_score=("weighted_score", "sum"),
            unweighted_risk_score=("llm_score", "sum"),
            max_weighted_score=("severity_weight", "sum"),
            hallucination_count=(
                "validation_category_norm",
                lambda s: (s == "H").sum()
            ),
            validation_denominator=(
                "validation_category_norm",
                lambda s: s.isin(["VC", "MC", "H"]).sum()
            ),
            indicator_count=("indicator_id", "nunique"),
            total_chunks=("total_chunks", "max"),
            successful_chunks=("successful_chunks", "max"),
            malformed_chunks=("malformed_chunks", "max"),
            chunk_coverage=("chunk_coverage", "max")
        )
    )

    site_model["weighted_risk_ratio"] = (
        site_model["weighted_risk_score"]
        /
        site_model["max_weighted_score"].replace(0, pd.NA)
    )

    site_model["risk_tier"] = (
        site_model["weighted_risk_score"]
        .apply(add_risk_tier)
    )

    site_model["hallucination_rate"] = (
        site_model["hallucination_count"]
        /
        site_model["validation_denominator"].replace(0, pd.NA)
    )

    # =====================================================
    # RISK SCORE DIVERGENCE ACROSS MODELS
    # =====================================================

    risk_divergence = (
        site_model.groupby(
            [
                "run_type",
                "run_no",
                "site_name",
                "domain"
            ],
            as_index=False
        )
        .agg(
            min_unweighted_risk_score=("unweighted_risk_score", "min"),
            max_unweighted_risk_score=("unweighted_risk_score", "max"),
            min_weighted_risk_score=("weighted_risk_score", "min"),
            max_weighted_risk_score=("weighted_risk_score", "max"),
            n_models=("model", "nunique")
        )
    )

    risk_divergence["unweighted_risk_score_range"] = (
        risk_divergence["max_unweighted_risk_score"]
        - risk_divergence["min_unweighted_risk_score"]
    )

    risk_divergence["weighted_risk_score_range"] = (
        risk_divergence["max_weighted_risk_score"]
        - risk_divergence["min_weighted_risk_score"]
    )

    risk_divergence["meaningful_unweighted_divergence"] = (
        risk_divergence["unweighted_risk_score_range"]
        > 3
    )

    # =====================================================
    # INDICATOR-LEVEL METRICS
    # =====================================================

    indicator_model = (
        df.groupby(
            [
                "run_type",
                "run_no",
                "indicator_id",
                "indicator_name",
                "model"
            ],
            as_index=False
        )
        .agg(
            mean_score=("llm_score", "mean"),
            total_flags=("llm_score", "sum"),
            hallucination_count=(
                "validation_category_norm",
                lambda s: (s == "H").sum()
            ),
            validation_denominator=(
                "validation_category_norm",
                lambda s: s.isin(["VC", "MC", "H"]).sum()
            )
        )
    )

    indicator_model["hallucination_rate"] = (
        indicator_model["hallucination_count"]
        /
        indicator_model["validation_denominator"].replace(0, pd.NA)
    )

    # =====================================================
    # HALLUCINATION RATE FROM HUMAN VALIDATION CATEGORIES
    # Formula: H / (VC + MC + H). AE is excluded.
    # =====================================================

    hallucination_by_model = summarize_hallucination_rate(
        df,
        ["run_type", "run_no", "model"]
    )

    hallucination_by_indicator = summarize_hallucination_rate(
        df,
        ["run_type", "run_no", "model", "indicator_id", "indicator_name"]
    )

    validation_category_counts = summarize_hallucination_rate(
        df,
        ["run_type", "run_no", "model", "validation_category_norm"]
    )

    # =====================================================
    # HUMAN VALIDATION / ALIGNMENT
    # =====================================================

    human_alignment_by_model = summarize_human_alignment(
        df,
        ["run_type", "run_no", "model"]
    )

    human_alignment_by_indicator = summarize_human_alignment(
        df,
        ["run_type", "run_no", "model", "indicator_id", "indicator_name"]
    )

    validation_notes = df[
        df.get("validation_note", pd.Series(index=df.index, dtype="object"))
        .notna()
    ][[
        "run_type",
        "run_no",
        "site_name",
        "domain",
        "model",
        "indicator_id",
        "llm_score",
        "human_score",
        "validation_category",
        "validation_category_norm",
        "validation_note",
        "evidence_quote"
    ]].copy() if "validation_note" in df.columns else pd.DataFrame()

    # =====================================================
    # PROCESSING METRICS
    # =====================================================

    processing_stats = (
        df.groupby(
            ["run_type", "run_no", "model"],
            as_index=False
        )
        .agg(
            avg_chunk_count=("total_chunks", "mean"),
            avg_chunk_coverage=("chunk_coverage", "mean"),
            total_hallucinations=(
                "validation_category_norm",
                lambda s: (s == "H").sum()
            ),
            validation_denominator=(
                "validation_category_norm",
                lambda s: s.isin(["VC", "MC", "H"]).sum()
            ),
            indicator_rows=("indicator_id", "count")
        )
    )

    processing_stats["hallucination_rate"] = (
        processing_stats["total_hallucinations"]
        /
        processing_stats["validation_denominator"].replace(0, pd.NA)
    )

    # =====================================================
    # MODEL DISAGREEMENT / INDICATOR-LEVEL DIVERGENCE
    # =====================================================

    pivot = df.pivot_table(
        index=[
            "run_type",
            "run_no",
            "site_name",
            "indicator_id"
        ],
        columns="model",
        values="llm_score",
        aggfunc="first"
    ).reset_index()

    model_cols = [
        col for col in pivot.columns
        if col not in [
            "run_type",
            "run_no",
            "site_name",
            "indicator_id"
        ]
    ]

    if model_cols:
        pivot["model_disagreement"] = (
            pivot[model_cols]
            .nunique(axis=1)
            > 1
        )

        disagreement = (
            pivot.groupby(
                ["run_type", "run_no", "indicator_id"],
                as_index=False
            )
            .agg(
                disagreement_cases=("model_disagreement", "sum"),
                total_cases=("model_disagreement", "count")
            )
        )

        disagreement["disagreement_rate"] = (
            disagreement["disagreement_cases"]
            /
            disagreement["total_cases"].replace(0, pd.NA)
        )
    else:
        disagreement = pd.DataFrame()

    # =====================================================
    # PAIRWISE MODEL AGREEMENT
    # =====================================================

    pairwise_rows = []

    for run_key, part in pivot.groupby(["run_type", "run_no"]):
        run_type, run_no = run_key

        for i, model_a in enumerate(model_cols):
            for model_b in model_cols[i + 1:]:
                valid = part[[model_a, model_b]].dropna()

                agreement_value = (
                    (valid[model_a] == valid[model_b]).mean()
                    if len(valid)
                    else pd.NA
                )

                pairwise_rows.append({
                    "run_type": run_type,
                    "run_no": run_no,
                    "model_a": model_a,
                    "model_b": model_b,
                    "pairwise_agreement": agreement_value,
                    "n_cases": len(valid)
                })

    pairwise_agreement = pd.DataFrame(pairwise_rows)

    # =====================================================
    # FLEISS KAPPA
    # =====================================================

    fleiss_rows = []

    for run_key, run_part in pivot.groupby(["run_type", "run_no"]):
        run_type, run_no = run_key

        for indicator_id, part in run_part.groupby("indicator_id"):
            ratings = part[model_cols].dropna(axis=0)

            kappa = (
                fleiss_kappa_binary(ratings)
                if len(model_cols) > 1
                else pd.NA
            )

            fleiss_rows.append({
                "run_type": run_type,
                "run_no": run_no,
                "indicator_id": indicator_id,
                "fleiss_kappa": kappa,
                "n_sites": len(ratings),
                "n_models": len(model_cols)
            })

    fleiss_kappa = pd.DataFrame(fleiss_rows)

    # =====================================================
    # TRUSTWORTHINESS INPUTS
    # Stability stays in the repeat-run workbook.
    # =====================================================

    trustworthiness_profile = hallucination_by_model[[
        "run_type",
        "run_no",
        "model",
        "hallucination_rate",
        "groundedness_rate"
    ]].copy()

    if not human_alignment_by_model.empty:
        trustworthiness_profile = trustworthiness_profile.merge(
            human_alignment_by_model[[
                "run_type",
                "run_no",
                "model",
                "human_alignment_rate",
                "human_validated_rows"
            ]],
            on=["run_type", "run_no", "model"],
            how="left"
        )
    else:
        trustworthiness_profile["human_alignment_rate"] = pd.NA
        trustworthiness_profile["human_validated_rows"] = 0

    trustworthiness_profile["groundedness_pass_gt_90pct"] = (
        trustworthiness_profile["groundedness_rate"]
        > 0.90
    )

    trustworthiness_profile["stability_from_repeat_workbook"] = pd.NA

    metrics = {
        "site_model_risk": site_model,
        "risk_score_divergence": risk_divergence,
        "indicator_model": indicator_model,
        "processing_stats": processing_stats,
        "disagreement": disagreement,
        "pairwise_agreement": pairwise_agreement,
        "fleiss_kappa": fleiss_kappa,
        "hallucination_by_model": hallucination_by_model,
        "hallucination_by_indicator": hallucination_by_indicator,
        "human_alignment_by_model": human_alignment_by_model,
        "human_alignment_by_indicator": human_alignment_by_indicator,
        "trustworthiness_profile": trustworthiness_profile,
        "validation_notes": validation_notes
    }

    if raw_df is not None and not raw_df.empty:
        raw = raw_df.copy()

        raw_processing = (
            raw.groupby(
                ["run_type", "run_no", "model"],
                as_index=False
            )
            .agg(
                total_attempts=("attempt_no", "count"),
                json_valid_attempts=("json_valid", "sum"),
                structure_valid_attempts=("structure_valid", "sum"),
                avg_output_size=("output_size", "mean"),
                avg_chunk_length=("chunk_length", "mean")
            )
        )

        raw_processing["malformed_json_or_structure_rate"] = (
            1
            -
            (
                raw_processing["structure_valid_attempts"]
                /
                raw_processing["total_attempts"].replace(0, pd.NA)
            )
        )

        metrics["raw_processing"] = raw_processing

    return metrics


In [73]:
# =========================================================
# BLOCK 39 - EXPORT METRICS
# =========================================================

def export_metrics(
    results_df,
    raw_df=None,
    filename="python_metrics_main.xlsx"
):

    metrics = compute_python_metrics(
        results_df=results_df,
        raw_df=raw_df
    )

    with pd.ExcelWriter(
        filename,
        engine="openpyxl"
    ) as writer:

        for sheet_name, metric_df in metrics.items():

            metric_df.to_excel(
                writer,
                sheet_name=sheet_name[:31],
                index=False
            )

    print(f"Metrics saved to {filename}")

    return metrics

In [74]:
# =========================================================
# BLOCK 40 - LOAD HUMAN-VALIDATED MAIN FILE AND RUN METRICS
# Repeat-run metrics are unchanged and still use the repeat files later.
# =========================================================

from pathlib import Path
from google.colab import files

HUMAN_VALIDATION_MAIN_FILE = "with_human_validation_results_main_all.xlsx"

if not Path(HUMAN_VALIDATION_MAIN_FILE).exists():
    print(
        "Upload the human-validated main results file. "
        "It will be saved as with_human_validation_results_main_all.xlsx"
    )

    uploaded_human_validation = files.upload()

    excel_files = [
        filename for filename in uploaded_human_validation.keys()
        if filename.lower().endswith((".xlsx", ".xls"))
    ]

    if not excel_files:
        raise ValueError("No Excel file uploaded for human validation results.")

    uploaded_file = excel_files[0]
    main_all_results = pd.read_excel(uploaded_file)

    main_all_results.to_excel(
        HUMAN_VALIDATION_MAIN_FILE,
        index=False
    )
else:
    main_all_results = pd.read_excel(
        HUMAN_VALIDATION_MAIN_FILE
    )

required_human_columns = [
    "human_score",
    "validation_category",
    "validation_note"
]

missing_human_columns = [
    col for col in required_human_columns
    if col not in main_all_results.columns
]

if missing_human_columns:
    print(
        "WARNING: missing human-validation columns:",
        missing_human_columns
    )
    print(
        "Metrics will still run, but human alignment or manual hallucination rates may be incomplete."
    )

try:
    main_all_raw = combine_main_raw_outputs()
except Exception as err:
    print("Raw output files not loaded; raw_processing sheet will be skipped.")
    print(err)
    main_all_raw = None

metrics = export_metrics(
    results_df=main_all_results,
    raw_df=main_all_raw,
    filename="python_metrics_main_with_human_validation.xlsx"
)

files.download(
    "python_metrics_main_with_human_validation.xlsx"
)


Metrics saved to python_metrics_main_with_visual_r17.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [75]:
# =========================================================
# BLOCK 41 - REPEAT STABILITY METRICS
# Uses:
# - main run
# - repeat run 01
# - repeat run 02
#
# R17 INCLUDED
# LLAMA excluded ONLY for R17
# =========================================================

def compute_repeat_stability(results_df):

    # =====================================================
    # PREPARE DATA
    # =====================================================

    df = results_df.copy()

    # exclude LLAMA ONLY for R17
    df = df[
        ~(
            (df["indicator_id"] == "R17")
            &
            (
                df["model"]
                .str.upper()
                .str.contains("LLAMA", na=False)
            )
        )
    ].copy()

    df["llm_score"] = pd.to_numeric(
        df["llm_score"],
        errors="coerce"
    ).fillna(0)

    # =====================================================
    # PIVOT RUNS
    # =====================================================

    pivot = df.pivot_table(

        index=[
            "site_name",
            "domain",
            "model",
            "indicator_id"
        ],

        columns="run_no",

        values="llm_score",

        aggfunc="first"

    ).reset_index()

    # =====================================================
    # STABILITY ACROSS 3 RUNS
    # =====================================================

    required_runs = [1, 2, 3]

    if all(run in pivot.columns for run in required_runs):

        pivot["stable"] = (

            (pivot[1] == pivot[2])

            &

            (pivot[1] == pivot[3])

        )

        pivot["stable"] = (
            pivot["stable"]
            .astype(int)
        )

    else:

        pivot["stable"] = pd.NA

    # =====================================================
    # STABILITY BY MODEL
    # =====================================================

    stability_by_model = (
        pivot.groupby(
            "model",
            as_index=False
        )
        .agg(

            stable_cases=(
                "stable",
                "sum"
            ),

            total_cases=(
                "stable",
                "count"
            )
        )
    )

    stability_by_model["stability_rate"] = (

        stability_by_model["stable_cases"]

        /

        stability_by_model["total_cases"]
        .replace(0, pd.NA)
    )

    # =====================================================
    # STABILITY BY INDICATOR
    # =====================================================

    stability_by_indicator = (
        pivot.groupby(
            [
                "model",
                "indicator_id"
            ],
            as_index=False
        )
        .agg(

            stable_cases=(
                "stable",
                "sum"
            ),

            total_cases=(
                "stable",
                "count"
            )
        )
    )

    stability_by_indicator["stability_rate"] = (

        stability_by_indicator["stable_cases"]

        /

        stability_by_indicator["total_cases"]
        .replace(0, pd.NA)
    )

    return (
        pivot,
        stability_by_model,
        stability_by_indicator
    )

In [76]:
# =========================================================
# BLOCK 42 - EXPORT REPEAT STABILITY
# =========================================================

def export_repeat_stability(
    results_df,
    filename="repeat_stability_with_visual_r17.xlsx"
):

    detail, by_model, by_indicator = (
        compute_repeat_stability(
            results_df
        )
    )

    with pd.ExcelWriter(
        filename,
        engine="openpyxl"
    ) as writer:

        detail.to_excel(
            writer,
            sheet_name="stability_detail",
            index=False
        )

        by_model.to_excel(
            writer,
            sheet_name="stability_by_model",
            index=False
        )

        by_indicator.to_excel(
            writer,
            sheet_name="stability_by_indicator",
            index=False
        )

    print(f"Repeat stability saved to {filename}")

    return (
        detail,
        by_model,
        by_indicator
    )

In [77]:
# =========================================================
# BLOCK 43 - LOAD ALL 3 RUNS
# =========================================================

main_run = pd.read_excel(
    "results_main_all_with_visual_r17.xlsx"
)

repeat_run_01 = pd.read_excel(
    "repeat_run_01_with_visual_r17.xlsx"
)

repeat_run_02 = pd.read_excel(
    "repeat_run_02_with_visual_r17.xlsx"
)

# =========================================================
# ASSIGN RUN NUMBERS
# =========================================================

main_run["run_no"] = 1

repeat_run_01["run_no"] = 2

repeat_run_02["run_no"] = 3

# =========================================================
# KEEP ONLY REPEATED SITES
# =========================================================

repeat_sites = repeat_run_01["domain"].unique()

main_run = main_run[
    main_run["domain"].isin(repeat_sites)
]

# =========================================================
# COMBINE ALL RUNS
# =========================================================

all_runs = pd.concat(

    [
        main_run,
        repeat_run_01,
        repeat_run_02
    ],

    ignore_index=True
)

# =========================================================
# RUN STABILITY ANALYSIS
# =========================================================

detail, by_model, by_indicator = (
    export_repeat_stability(

        results_df=all_runs,

        filename="repeat_stability_with_visual_r17.xlsx"
    )
)
# =========================================================
# DOWNLOAD FILE
# =========================================================

from google.colab import files

files.download(
    "repeat_stability_with_visual_r17.xlsx"
)

Repeat stability saved to repeat_stability_with_visual_r17.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>